# 8 · Re-train the 3 heads at SENTENCE level, then fuse (Option B)

The Option-A heads were trained on **2-word** TriviaQA answers (BLEURT string-match) — they track answer *form*, not sentence factuality, so the demo over-flags full sentences. Here we retrain each head on the **same regime the live demo uses** — ONE forced factual sentence per question (Instruct chat template) — with cheap, accurate **reference-match** labels (the answer's known aliases). Train == inference, so the per-sentence scores are finally in-distribution.

**Method (the right order): make each technique work FIRST, then fuse.**
1. **Config** — every knob in one cell.
2. **Build dataset** *(GPU)* — generate one sentence per question; cache RAW SEP (135168-d) + HalluShift (71-d) features + `(question, answer)` for TSV; label by reference match; drop refusals ("I don't know" is not a claim). Cached so head re-training needs no GPU re-gen.
3. **Re-train each head + per-head HELD-OUT AUROC** *(CPU + 1 GPU for TSV)* — the **decisive checkpoint**: if a head separates (AUROC >~0.65) it's worth fusing; if all stay ~0.5 that's the honest finding.
4. **Fuse + evaluate** *(CPU)* — only if the heads separate: fuse the 3 scores on the SAME held-out split (test rows scored by heads that never saw them — no leakage); AUROC/AUPR/F1 + confusion + ROC/PR.
5. **Smoke-test the product** *(GPU)* — flag each sentence of a fresh answer via the Option-B heads.

**Re-running with different settings = edit the CONFIG cell only.** Live progress bars throughout.

*Honest note:* empirical run — no guaranteed accuracy; the per-head AUROC cell **measures** whether the signals carry sentence-level factuality before we invest in fusion.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
# ===== CONFIG (edit here) ==================================================================
TAG       = 's1'           # tag -> artifacts/*/*_sentence_<TAG>.*, data/claims_<TAG>.parquet, fusion_claim_<TAG>.pkl
DATASETS  = ['triviaqa']   # named-entity QA where reference-match works; add 'nq_open' for more data
N         = 1500           # questions per dataset
OFFSET    = 0              # start index into the dataset
MAX_NEW_TOKENS = 64        # one factual sentence
EPOCHS_TSV = 40            # TSV steering-vector epochs
C         = 0.5            # fusion L2 regularization
# ==========================================================================================
print('config set; TAG =', TAG, '| datasets =', DATASETS, '| N =', N)

### 2 · Build the sentence dataset — **GPU pass** (generate + RAW features + reference labels)
Generates one factual sentence per question, caches RAW per-head features + reference-match labels, drops refusals. Saves `data/claims_sent_<TAG>.parquet` + `_sepfeats.npy` so step 3 can re-run heads **without** re-generating. Start with a small `N` to smoke-test before scaling.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
from train_claim_heads import build
df, sep_feats = build(tag=TAG, datasets=DATASETS, n=N, offset=OFFSET, max_new_tokens=MAX_NEW_TOKENS)
print('label balance (0=truthful, 1=halluc):', df['hallucination'].value_counts().to_dict())
df[['question','answer','hallucination','source']].head(8)

### 3 · Re-train each head + **per-head HELD-OUT AUROC** — the decisive checkpoint *(CPU + 1 GPU for TSV)*
ONE shared 75/25 split; refits the SEP probe + HalluShift MLP + TSV vector on TRAIN and scores the held-out TEST. **Read the table before going further:** a head with AUROC >~0.65 carries sentence-level factuality and is worth fusing; if all three hover ~0.5, these signals don't separate sentences and fusion won't rescue them (the honest finding). Saves the `_sentence_<TAG>` head artifacts + a scored `data/claims_<TAG>.parquet`.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
from train_claim_heads import train_heads
H = train_heads(tag=TAG, epochs_tsv=EPOCHS_TSV)
H['summary']

### 4 · Fuse + evaluate *(CPU)* — run only if at least one head separated
Fuses the 3 scores using the SAME split (TEST rows are out-of-sample for the heads → no leakage); thresholds picked on TRAIN. Prints per-detector vs FUSED AUROC/AUPR/F1 and saves `models/fusion_claim_<TAG>.pkl` (+ thresholds).

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..','tools'))
from train_claim_fusion import train
R = train(tag=TAG, C=C)
R['summary']

### Confusion matrices + ROC / PR

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt, metrics as M
res = R['metrics']
fig, ax = plt.subplots(1, 2, figsize=(11,4)); M.plot_roc(ax[0], res); M.plot_pr(ax[1], res)
fig.suptitle('Per-sentence detection — ROC / PR'); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 4, figsize=(15,3.4))
for axx,(name,m) in zip(axes, res.items()):
    M.plot_confusion(axx, m['confusion_matrix'], title=f"{name}\nF1={m['F1']:.2f}")
plt.tight_layout(); plt.show()

### Thresholds + a note on localization
Training is one sentence per question, so within-passage 'find-the-wrong-sentence' is **not** measured on this set (each prompt has a single claim) — it is demonstrated on a real multi-sentence answer in the smoke test below.

In [ ]:
loc = R['localization']
if loc['n_multiclaim_prompts'] > 0:
    print(f"Find-the-wrong-sentence (top-1): {loc['localization_top1']:.3f} over "
          f"{loc['n_multiclaim_prompts']} multi-claim passages | within-passage AUROC "
          f"{loc['within_passage_auroc']:.3f}")
else:
    print('within-passage localization not measured here (one claim per question) — see smoke test')
print(f"Calibrated thresholds: T_MED={R['t_med']}  T_HIGH={R['t_high']}")

### 5 · Smoke-test the product — **GPU** — Option-B heads, per-sentence flags
Loads the `_sentence_<TAG>` heads (`sentence_tag=TAG` → sentence-regime generation, train==inference) and the per-claim fusion, then flags each claim sentence of a fresh answer. Obscure entities elicit more hallucinations.

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
import torch, gc, json
from pipeline import HallKingPipeline
from fusion import FusionModel
from localize import localize
gc.collect(); torch.cuda.empty_cache()
pipe = HallKingPipeline(dataset='triviaqa', separate_tsv=False, sentence_tag=TAG).load()
pipe.fusion = FusionModel.load(os.path.join('..','models',f'fusion_claim_{TAG}.pkl'))
THR = json.load(open(os.path.join('..','models',f'fusion_claim_{TAG}_thresholds.json')))
for Q in ['Who painted the Mona Lisa?', 'What is the capital of Australia?',
          'Tell me about the Sigiriya rock fortress.']:
    res = localize(pipe, Q, max_new_tokens=80, use_claim_filter=True)
    print('Q:', Q); print('A:', res['answer'])
    for s in res['sentences']:
        if s['fused'] is None:
            tg='filler   '; fv='   '
        else:
            fv=f"{s['fused']:.2f}"
            tg = 'HALLUC   ' if s['fused']>=THR['t_high'] else ('UNCERTAIN' if s['fused']>=THR['t_med'] else 'ok       ')
        print(f"   [{tg} {fv}] {s['sentence']}")
    print()

### Wire it into the live demo (after the numbers check out)
The pipeline already supports Option B: `HallKingPipeline(..., sentence_tag='<TAG>')` loads the `_sentence_<TAG>` heads and generates in the sentence regime. To switch the web demo, set `SENTENCE_TAG = '<TAG>'` in `backend/app.py` (it loads `models/fusion_claim_<TAG>.pkl` + thresholds). Until the numbers are good, the backend stays on the Option-A short-QA path.